# 02 - M2: frozen-CLIP probes (gate: reproduce Chethan's AUROCs on the controlled protocol)
**Runtime > T4 GPU.** First run extracts CLIP features for ~10k images (~10-20 min) and caches them to Drive; later runs reuse the cache.
Gate: our AUROC point estimate must fall inside Chethan's 95% CI for each of the 6 (fold, probe) cells.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'main'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import numpy as np, torch
from datasets import load_dataset
from transformers import CLIPVisionModel
from src.data import load_manifest
from src.features import ClipTaps, RowDataset, extract_to_cache, FeatureStore, union_rows

PROTOCOL = 'controlled'          # the M2 gate protocol; set 'raw' for a secondary run
FOLDS = ['Midjourney', 'BigGAN', 'ADM']
manifests = [load_manifest(f) for f in FOLDS]
rev = manifests[0]['revision']; assert all(m['revision'] == rev for m in manifests)
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=rev)
clip = CLIPVisionModel.from_pretrained('openai/clip-vit-large-patch14-336', torch_dtype=torch.float32).cuda()
enc = ClipTaps(clip, taps=(16, 24))
assert clip.config.num_hidden_layers == 24 and clip.config.image_size == 336
print('CLIP loaded')

In [ ]:
# Extract once (resumable: re-running skips finished chunks). train rows -> HF 'train'; test rows -> HF 'validation'.
FEAT = f'{DRIVE_ROOT}/features/{PROTOCOL}'
for split, which, hf in (('train', 'train', ds['train']), ('validation', 'test', ds['validation'])):
    rows = sorted(set(union_rows(manifests, 'train') + union_rows(manifests, 'val'))) if which == 'train' else union_rows(manifests, 'test')
    print(split, len(rows), 'rows')
    extract_to_cache(enc, RowDataset(hf, rows, PROTOCOL), f'{FEAT}/{split}', chunk=512, batch_size=32,
                     num_workers=2, device='cuda', autocast=True)

In [ ]:
# Probes on cached features (CPU, seconds). C is chosen on val (seen generators only); test is scored once.
import json, pandas as pd
from src.probes import run_fold, gate_report
tr, te = FeatureStore(f'{FEAT}/train'), FeatureStore(f'{FEAT}/validation')
results = [run_fold(m, tr, te, kind) for m in manifests for kind in ('univfd', 'multilayer')]
rep = pd.DataFrame(gate_report(results)); display(rep)
print('GATE PASS' if rep.in_ref_ci.all() else 'GATE FAIL: see rows with in_ref_ci False')

In [ ]:
# Save results (Drive + repo runs/ folder). Then add rows to docs/experiment_log.md and commit runs/*.json.
out = {'protocol': PROTOCOL, 'revision': rev, 'results': results, 'gate': rep.to_dict('records')}
for p in (f'{DRIVE_ROOT}/runs/m2_{PROTOCOL}.json', f'/content/csm-clip/runs/m2_{PROTOCOL}.json'):
    json.dump(out, open(p, 'w'), indent=1)
print(rep[['fold','kind','ours','ours_ci','ref','ref_ci','in_ref_ci']].to_string(index=False))